# Warfarin Pharmacogenomics: 7-Model Comparison Pipeline
**Linear and Machine Learning Models for Pharmacogenomic Warfarin Dose Prediction**

Comparative Performance and Transportability to a Held-Out Asian Cohort

**Authors:** Amir Sams, Saurabh Kumar Chauhan, Maliha Khan, Suruchi Singh  
**Affiliation:** Department of Pharmacology, School of Pharmaceutical Education and Research, Jamia Hamdard, New Delhi 110062, India

---
This notebook implements the complete analytical pipeline for a seven-model comparison study using the IWPC (International Warfarin Pharmacogenetics Consortium) dataset from PharmGKB.

**Models:** Multiple Linear Regression (MLR), Ridge, Lasso, Support Vector Regression (RBF kernel), Random Forest (untuned), Random Forest (tuned), XGBoost (tuned)

**Evaluation cohorts:** Non-Asian hold-out test set (30%) and Asian transportability cohort (held out entirely)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, KFold, RandomizedSearchCV)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from scipy import stats

SEED = 42
np.random.seed(SEED)
rng = np.random.RandomState(SEED)

xls = pd.ExcelFile('/content/drive/MyDrive/warfarin.xls')
df_raw = xls.parse('Subject Data')
print(f"Loaded: {len(df_raw)} rows")

## 1. Data Cleaning & Exclusion Cascade
Complete-case analysis requiring: stable dose achieved, non-missing therapeutic dose (<300 mg/week), CYP2C9 consensus, VKORC1 -1639 consensus, height, weight, and age.

In [ ]:
df = df_raw.copy()
n0 = len(df)
df = df[df['Subject Reached Stable Dose of Warfarin'] == 1]
print(f"Stable dose reached: {len(df)} (excluded {n0 - len(df)})")
n1 = len(df)
df = df[df['Therapeutic Dose of Warfarin'].notna()]
print(f"Dose not missing: {len(df)} (excluded {n1 - len(df)})")
n2 = len(df)
df = df[df['Therapeutic Dose of Warfarin'] < 300]
print(f"Dose < 300 mg/wk: {len(df)} (excluded {n2 - len(df)})")
n3 = len(df)
df = df[df['CYP2C9 consensus'].notna()]
print(f"CYP2C9 present: {len(df)} (excluded {n3 - len(df)})")
n4 = len(df)
df = df[df['VKORC1 -1639 consensus'].notna()]
print(f"VKORC1 present: {len(df)} (excluded {n4 - len(df)})")
n5 = len(df)
df = df[df['Height (cm)'].notna()]
print(f"Height present: {len(df)} (excluded {n5 - len(df)})")
n6 = len(df)
df = df[df['Weight (kg)'].notna()]
print(f"Weight present: {len(df)} (excluded {n6 - len(df)})")
n7 = len(df)
df = df[df['Age'].notna()]
print(f"Age present: {len(df)} (excluded {n7 - len(df)})")

def age_to_ordinal(age_band):
    try:
        lower = int(str(age_band).split('-')[0].strip().split('+')[0])
        return lower // 10
    except:
        return np.nan

df['age_ordinal'] = df['Age'].apply(age_to_ordinal)
df = df[df['age_ordinal'].notna()]
print(f"\nFinal analytical cohort: {len(df)}")

inducer_cols = ['Carbamazepine (Tegretol)', 'Phenytoin (Dilantin)', 'Rifampin or Rifampicin']
df[inducer_cols] = df[inducer_cols].fillna(0)
df['enzyme_inducer'] = (df[inducer_cols].sum(axis=1) > 0).astype(int)
df['Amiodarone (Cordarone)'] = df['Amiodarone (Cordarone)'].fillna(0)

## 2. Cohort Partitioning
Asian patients (by OMB race classification) are held out entirely as a transportability evaluation cohort. The remaining non-Asian cohort is split 70/30 for training and testing.

In [ ]:
race_omb = df['Race (OMB)'].astype(str).str.strip().str.lower()
asian_mask = race_omb == 'asian'
df_asian = df[asian_mask].copy()
df_main = df[~asian_mask].copy()
print(f"Asian held-out: {len(df_asian)}")
print(f"Non-Asian cohort: {len(df_main)}")

predictors = ['age_ordinal', 'Height (cm)', 'Weight (kg)',
              'CYP2C9 consensus', 'VKORC1 -1639 consensus',
              'Race (OMB)', 'enzyme_inducer', 'Amiodarone (Cordarone)']

def build_X(data):
    return pd.get_dummies(data[predictors],
                          columns=['CYP2C9 consensus', 'VKORC1 -1639 consensus', 'Race (OMB)'])

X_main = build_X(df_main)
y_main_sqrt = np.sqrt(df_main['Therapeutic Dose of Warfarin'])

X_train, X_test, y_train, y_test = train_test_split(
    X_main, y_main_sqrt, test_size=0.3, random_state=SEED)

y_test_actual = y_test.values ** 2
y_asian_actual = df_asian['Therapeutic Dose of Warfarin'].values

X_asian = build_X(df_asian).reindex(columns=X_train.columns, fill_value=0)
y_asian_sqrt = np.sqrt(df_asian['Therapeutic Dose of Warfarin'])

print(f"Train: {X_train.shape} | Test: {X_test.shape} | Asian: {X_asian.shape}")
feature_names = X_train.columns.tolist()
print(f"Features ({len(feature_names)}): {feature_names}")

## 3. Train All 7 Models

| Model | Type | Tuning |
|---|---|---|
| MLR | Linear | None |
| Ridge | Linear | α = 1.0 |
| Lasso | Linear | α = 0.01 |
| SVR (RBF) | Kernel | RandomizedSearchCV (30 iter, 5-fold) |
| RF (untuned) | Ensemble | n_estimators=500 |
| RF (tuned) | Ensemble | RandomizedSearchCV (60 iter, 5-fold) |
| XGBoost (tuned) | Gradient boosting | RandomizedSearchCV (60 iter, 5-fold) |

All models predict √(dose); predictions are back-transformed by squaring. Hyperparameter selection uses neg_mean_absolute_error on the √(dose) scale within 5-fold CV on training data only.

In [ ]:
mlr = LinearRegression().fit(X_train, y_train)
print("[1/7] MLR trained")

ridge = Ridge(alpha=1.0).fit(X_train, y_train)
print("[2/7] Ridge trained")

lasso = Lasso(alpha=0.01, max_iter=10000).fit(X_train, y_train)
print("[3/7] Lasso trained")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_asian_scaled = scaler.transform(X_asian)

svr_param_dist = {
    'C': [0.1, 1, 10, 50, 100],
    'gamma': [0.001, 0.01, 0.05, 0.1, 'scale'],
    'epsilon': [0.05, 0.1, 0.2, 0.3, 0.5],
}
svr_search = RandomizedSearchCV(
    SVR(kernel='rbf'), svr_param_dist,
    n_iter=30, cv=5, scoring='neg_mean_absolute_error',
    random_state=SEED, n_jobs=-1)
svr_search.fit(X_train_scaled, y_train)
svr_model = svr_search.best_estimator_
print(f"[4/7] SVR trained \u2014 best: {svr_search.best_params_}")

In [ ]:
rf_untuned = RandomForestRegressor(n_estimators=500, random_state=SEED)
rf_untuned.fit(X_train, y_train)
print("[5/7] RF (untuned) trained")

rf_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 8, 10, 15, None],
    'min_samples_leaf': [1, 2, 4],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', 0.5],
}
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=SEED), rf_param_dist,
    n_iter=60, cv=5, scoring='neg_mean_absolute_error',
    random_state=SEED, n_jobs=-1)
rf_search.fit(X_train, y_train)
rf_tuned = rf_search.best_estimator_
print(f"[6/7] RF (tuned) trained \u2014 best: {rf_search.best_params_}")

from xgboost import XGBRegressor
xgb_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5, 7],
}
xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=SEED, verbosity=0), xgb_param_dist,
    n_iter=60, cv=5, scoring='neg_mean_absolute_error',
    random_state=SEED, n_jobs=-1)
xgb_search.fit(X_train, y_train)
xgb_tuned = xgb_search.best_estimator_
print(f"[7/7] XGBoost trained \u2014 best: {xgb_search.best_params_}")

## 4. Table 3 — Model Performance (All 7 Models × 2 Cohorts)

In [ ]:
def get_predictions(model, is_svr=False):
    if is_svr:
        pred_test = model.predict(X_test_scaled) ** 2
        pred_asian = model.predict(X_asian_scaled) ** 2
    else:
        pred_test = model.predict(X_test) ** 2
        pred_asian = model.predict(X_asian) ** 2
    return np.maximum(pred_test, 0), np.maximum(pred_asian, 0)

models_dict = {
    'MLR': (mlr, False),
    'Ridge': (ridge, False),
    'Lasso': (lasso, False),
    'SVR (RBF)': (svr_model, True),
    'RF (untuned)': (rf_untuned, False),
    'RF (tuned)': (rf_tuned, False),
    'XGBoost (tuned)': (xgb_tuned, False),
}

preds_main = {}
preds_asian = {}
table3_rows = []

for name, (model, is_svr) in models_dict.items():
    pred_m, pred_a = get_predictions(model, is_svr)
    preds_main[name] = pred_m
    preds_asian[name] = pred_a
    
    mae_m = mean_absolute_error(y_test_actual, pred_m)
    rmse_m = np.sqrt(mean_squared_error(y_test_actual, pred_m))
    r2_m = r2_score(y_test_actual, pred_m)
    p20_m = 100 * np.mean(np.abs(y_test_actual - pred_m) <= 0.20 * y_test_actual)
    
    mae_a = mean_absolute_error(y_asian_actual, pred_a)
    rmse_a = np.sqrt(mean_squared_error(y_asian_actual, pred_a))
    r2_a = r2_score(y_asian_actual, pred_a)
    p20_a = 100 * np.mean(np.abs(y_asian_actual - pred_a) <= 0.20 * y_asian_actual)
    
    table3_rows.append({
        'Model': name,
        'Main MAE': f'{mae_m:.2f}', 'Main RMSE': f'{rmse_m:.2f}',
        'Main R\u00b2': f'{r2_m:.3f}', 'Main P20%': f'{p20_m:.1f}',
        'Asian MAE': f'{mae_a:.2f}', 'Asian RMSE': f'{rmse_a:.2f}',
        'Asian R\u00b2': f'{r2_a:.3f}', 'Asian P20%': f'{p20_a:.1f}',
    })

table3 = pd.DataFrame(table3_rows)
print("TABLE 3: Model Performance")
print("=" * 100)
print(table3.to_string(index=False))

## 5. Bootstrap 95% Confidence Intervals (2,000 resamples)
Bootstrap CIs for MAE and P₂₀ for all seven models on both evaluation cohorts.

In [ ]:
N_BOOT = 2000

def calc_mae(y, p): return np.mean(np.abs(y - p))
def calc_p20(y, p): return 100 * np.mean(np.abs(y - p) <= 0.20 * y)

def bootstrap_metric(y_true, y_pred, metric_fn, n_boot=N_BOOT):
    n = len(y_true)
    vals = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        vals.append(metric_fn(y_true[idx], y_pred[idx]))
    point = metric_fn(y_true, y_pred)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return point, lo, hi

for label, y_true, preds in [("Main test set", y_test_actual, preds_main),
                               ("Asian cohort", y_asian_actual, preds_asian)]:
    print(f"\n{label} (n={len(y_true)}):")
    print(f"{'Model':<20s} {'MAE [95% CI]':>28s}  {'P20% [95% CI]':>28s}")
    print("-" * 80)
    for name, pred in preds.items():
        mae_p, mae_lo, mae_hi = bootstrap_metric(y_true, pred, calc_mae)
        p20_p, p20_lo, p20_hi = bootstrap_metric(y_true, pred, calc_p20)
        print(f"{name:<20s} {mae_p:>6.2f} [{mae_lo:>6.2f}, {mae_hi:>6.2f}]"
              f"  {p20_p:>6.1f}% [{p20_lo:>5.1f}, {p20_hi:>5.1f}]")

## 6. Paired Bootstrap Comparisons (ΔMAE vs MLR)
For each bootstrap resample, the same patient indices are used for every model. Positive ΔMAE means the model performs worse than MLR.

In [ ]:
def paired_bootstrap(y_true, pred_ref, pred_comp, n_boot=N_BOOT):
    n = len(y_true)
    diffs = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        mae_ref = np.mean(np.abs(y_true[idx] - pred_ref[idx]))
        mae_comp = np.mean(np.abs(y_true[idx] - pred_comp[idx]))
        diffs.append(mae_comp - mae_ref)
    diffs = np.array(diffs)
    return np.mean(diffs), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

for label, y_true, preds in [("Main test set", y_test_actual, preds_main),
                               ("Asian cohort", y_asian_actual, preds_asian)]:
    ref = preds['MLR']
    print(f"\n{label} (reference: MLR):")
    print(f"{'Comparison':<30s} {'\u0394MAE':>8s} {'95% CI':>20s} {'Significant?':>14s}")
    print("-" * 75)
    for name, pred in preds.items():
        if name == 'MLR': continue
        mean_d, lo, hi = paired_bootstrap(y_true, ref, pred)
        sig = "YES *" if (lo > 0 or hi < 0) else "No"
        print(f"{name + ' vs MLR':<30s} {mean_d:>+8.3f} [{lo:>+7.3f}, {hi:>+7.3f}]   {sig:>10s}")

## 7. Calibration Analysis
Regress observed dose on predicted dose. Ideal calibration: intercept = 0, slope = 1.

In [ ]:
for label, y_true, preds in [("Main test set", y_test_actual, preds_main),
                               ("Asian cohort", y_asian_actual, preds_asian)]:
    print(f"\n{label}:")
    print(f"{'Model':<20s} {'Intercept [95% CI]':>30s}  {'Slope [95% CI]':>30s}")
    print("-" * 85)
    for name, pred in preds.items():
        slope, intercept, _, _, _ = stats.linregress(pred, y_true)
        slopes, intercepts = [], []
        n = len(y_true)
        for _ in range(N_BOOT):
            idx = rng.choice(n, size=n, replace=True)
            s, i, _, _, _ = stats.linregress(pred[idx], y_true[idx])
            slopes.append(s)
            intercepts.append(i)
        s_lo, s_hi = np.percentile(slopes, [2.5, 97.5])
        i_lo, i_hi = np.percentile(intercepts, [2.5, 97.5])
        print(f"{name:<20s} {intercept:>7.2f} [{i_lo:>7.2f}, {i_hi:>7.2f}]"
              f"  {slope:>7.3f} [{s_lo:>7.3f}, {s_hi:>7.3f}]")

## 8. Dose-Range Stratified Performance
Performance by clinically defined dose ranges: low (<21 mg/week), intermediate (21–49 mg/week), high (>49 mg/week).

In [ ]:
dose_ranges = [
    ('Low (<21 mg/wk)', 0, 21),
    ('Intermediate (21-49)', 21, 49),
    ('High (>49 mg/wk)', 49, float('inf')),
]

for label, y_true, preds in [("Main test set", y_test_actual, preds_main),
                               ("Asian cohort", y_asian_actual, preds_asian)]:
    print(f"\n{'='*60}")
    print(f"{label}")
    for range_label, lo_dose, hi_dose in dose_ranges:
        if hi_dose == float('inf'):
            mask = y_true > lo_dose
        else:
            mask = (y_true >= lo_dose) & (y_true < hi_dose)
        n_s = mask.sum()
        print(f"\n  {range_label} (n={n_s}):")
        print(f"  {'Model':<20s} {'MAE':>8s} {'P20%':>8s} {'Over20%':>8s} {'Under20%':>8s}")
        print("  " + "-" * 55)
        for name, pred in preds.items():
            if n_s == 0: continue
            y_s, p_s = y_true[mask], pred[mask]
            mae_s = np.mean(np.abs(y_s - p_s))
            p20_s = 100 * np.mean(np.abs(y_s - p_s) <= 0.20 * y_s)
            over = 100 * np.mean((p_s - y_s) > 0.20 * y_s)
            under = 100 * np.mean((y_s - p_s) > 0.20 * y_s)
            print(f"  {name:<20s} {mae_s:>8.2f} {p20_s:>7.1f}% {over:>7.1f}% {under:>7.1f}%")

## 9. Five-Fold Cross-Validation (Table S1)
Five-fold CV on the non-Asian training cohort. Scaling is fit within each fold for SVR to prevent data leakage.

In [ ]:
X_all = build_X(df_main).reset_index(drop=True)
y_all_sqrt = np.sqrt(df_main['Therapeutic Dose of Warfarin']).reset_index(drop=True)
dose_all = df_main['Therapeutic Dose of Warfarin'].values
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def cv_model(model_fn, needs_scaling=False):
    fold_maes, fold_p20s = [], []
    for tr, te in kf.split(X_all):
        X_tr, X_te = X_all.iloc[tr], X_all.iloc[te]
        y_tr = y_all_sqrt.iloc[tr]
        actual = dose_all[te]
        if needs_scaling:
            sc = StandardScaler()
            X_tr_s = sc.fit_transform(X_tr)
            X_te_s = sc.transform(X_te)
            m = model_fn().fit(X_tr_s, y_tr)
            pred = m.predict(X_te_s) ** 2
        else:
            m = model_fn().fit(X_tr, y_tr)
            pred = m.predict(X_te) ** 2
        pred = np.maximum(pred, 0)
        fold_maes.append(mean_absolute_error(actual, pred))
        fold_p20s.append(100 * np.mean(np.abs(actual - pred) <= 0.20 * actual))
    return fold_maes, fold_p20s

cv_configs = [
    ('MLR', lambda: LinearRegression(), False),
    ('Ridge', lambda: Ridge(alpha=1.0), False),
    ('Lasso', lambda: Lasso(alpha=0.01, max_iter=10000), False),
    ('SVR (RBF)', lambda: SVR(**svr_search.best_params_, kernel='rbf'), True),
    ('RF (untuned)', lambda: RandomForestRegressor(n_estimators=500, random_state=SEED), False),
    ('RF (tuned)', lambda: RandomForestRegressor(**rf_search.best_params_, random_state=SEED), False),
    ('XGBoost (tuned)', lambda: XGBRegressor(**xgb_search.best_params_, random_state=SEED, verbosity=0), False),
]

print(f"{'Model':<20s} {'MAE \u00b1 SD':>15s}  {'P20% \u00b1 SD':>15s}")
print("-" * 55)
for name, fn, scaled in cv_configs:
    maes, p20s = cv_model(fn, scaled)
    print(f"{name:<20s} {np.mean(maes):>6.2f} \u00b1 {np.std(maes, ddof=1):<5.2f}  "
          f"{np.mean(p20s):>6.1f} \u00b1 {np.std(p20s, ddof=1):<5.1f}")

## 10. Full Signed MLR Coefficients
Complete coefficient table with feature names and reference categories.

In [ ]:
print(f"{'Feature':<45s} {'Coefficient':>12s}")
print("-" * 60)
print(f"{'Intercept':<45s} {mlr.intercept_:>+12.6f}")
for fname, coef in sorted(zip(feature_names, mlr.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f"{fname:<45s} {coef:>+12.6f}")

## 11. Missingness Audit (Before Exclusions)
Variable-level missingness in the raw IWPC dataset before any exclusion criteria are applied.

In [ ]:
miss_cols = [
    'Subject Reached Stable Dose of Warfarin',
    'Therapeutic Dose of Warfarin',
    'CYP2C9 consensus', 'VKORC1 -1639 consensus',
    'Height (cm)', 'Weight (kg)', 'Age',
    'Race (OMB)', 'Race (Reported)',
    'Carbamazepine (Tegretol)', 'Phenytoin (Dilantin)',
    'Rifampin or Rifampicin', 'Amiodarone (Cordarone)',
]

print(f"{'Variable':<45s} {'Present':>8s} {'Missing':>8s} {'%':>6s}")
print("-" * 70)
n_total = len(df_raw)
for col in miss_cols:
    if col in df_raw.columns:
        n_miss = df_raw[col].isna().sum()
        n_present = n_total - n_miss
        print(f"{col:<45s} {n_present:>8d} {n_miss:>8d} {100*n_miss/n_total:>5.1f}%")

## 12. Indian Data-Completeness Cascade (Table 4)
Stepwise attrition of self-reported Indian patients through the inclusion criteria.

In [ ]:
raw = df_raw.copy()
indian = raw[raw['Race (Reported)'].astype(str).str.strip() == 'Indian']
print(f"Self-reported Indian: {len(indian)}")
step1 = indian[indian['Therapeutic Dose of Warfarin'].notna() &
               (indian['Subject Reached Stable Dose of Warfarin'] == 1)]
print(f"+ Valid stable dose: {len(step1)}")
step2 = step1[step1['CYP2C9 consensus'].notna()]
print(f"+ CYP2C9 genotype: {len(step2)}")
step3 = step2[step2['VKORC1 -1639 consensus'].notna()]
print(f"+ VKORC1 genotype: {len(step3)}")
step4 = step3[step3['Height (cm)'].notna() & step3['Weight (kg)'].notna() &
              step3['Age'].notna()]
print(f"+ Complete demographics: {len(step4)}")

## 13. Sensitivity: Race-Agnostic Transportability
Re-train models without any race features and evaluate on the Asian cohort.

In [ ]:
pred_norace = ['age_ordinal', 'Height (cm)', 'Weight (kg)',
               'CYP2C9 consensus', 'VKORC1 -1639 consensus',
               'enzyme_inducer', 'Amiodarone (Cordarone)']
def build_X_norace(data):
    return pd.get_dummies(data[pred_norace],
                          columns=['CYP2C9 consensus', 'VKORC1 -1639 consensus'])

def eval_asian(build_fn, label):
    Xall = build_fn(df_main)
    yall = np.sqrt(df_main['Therapeutic Dose of Warfarin'])
    Xtr, _, ytr, _ = train_test_split(Xall, yall, test_size=0.3, random_state=42)
    Xas = build_fn(df_asian).reindex(columns=Xtr.columns, fill_value=0)
    yas = df_asian['Therapeutic Dose of Warfarin'].values
    for nm, ctor in [("MLR", lambda: LinearRegression()),
                     ("RF (500)", lambda: RandomForestRegressor(n_estimators=500, random_state=42))]:
        m = ctor().fit(Xtr, ytr)
        pred = m.predict(Xas) ** 2
        mae = mean_absolute_error(yas, pred)
        pct = np.mean(np.abs(yas - pred) <= 0.2 * yas) * 100
        print(f"[{label:9}] {nm}: Asian MAE {mae:.2f}, P20 {pct:.1f}%")

print("=== TRANSPORTABILITY: WITH vs WITHOUT RACE ===")
eval_asian(build_X, "with race")
eval_asian(build_X_norace, "no race")

## 14. Selected Hyperparameters (Table 2)

In [ ]:
print("RF (tuned):", rf_search.best_params_)
print("XGBoost:   ", xgb_search.best_params_)
print("SVR:       ", svr_search.best_params_)

## 15. Software Versions

In [ ]:
import sklearn, scipy
print(f"Python {sys.version.split()[0]}")
print(f"scikit-learn {sklearn.__version__}")
print(f"pandas {pd.__version__}")
print(f"numpy {np.__version__}")
print(f"scipy {scipy.__version__}")
try:
    import xgboost as xgb_lib
    print(f"XGBoost {xgb_lib.__version__}")
except: pass